# Setup

## Load necessary packages

In [2]:
import pandas as pd
pd.options.mode.chained_assignment = None
import matplotlib.pyplot as plt
import numpy as np

In [3]:
sim_dir = "results/for_real_comp"

# Flipper validation

## Load data

In [4]:
# Load up the true binding sites. 
true_sites = pd.read_csv(f"{sim_dir}/true_values/binding_metrics.tsv", sep = "\t")

# Load in the skipper windows. 
skipper_windows = pd.read_csv(f"{sim_dir}/flipper/intermediates/window_counts_INf.tsv", sep = "\t")

# Select skipper windows.
true_sites_skipper = true_sites[
    true_sites["window_name"].isin(skipper_windows["window_name"])
]

# Load flipper windows.
Flipper_results = pd.read_csv(f"{sim_dir}/flipper/tables/all_windows.tsv", sep = "\t")

## Identify true positives/negatives

In [6]:
# Select whether or not there was true change for a given site. 
true_sites_skipper['change_true'] = np.where(
    true_sites_skipper['trueZeta'] >  1.5, 'increased',
    np.where(
        true_sites_skipper['trueZeta'] < 0.66, 'decreased',
        'no_change'
    )
)

# Get the Flipper predicted change for each site. 
Flipper_results['change_predict'] = np.where(
    (Flipper_results['padj'] < 0.05) & (Flipper_results['log2FoldChange'] > 0), 'increased',
    np.where(
        (Flipper_results['padj'] < 0.05) & (Flipper_results['log2FoldChange'] < 0), 'decreased',
        'no_change'
    )
)

In [7]:
merged = (true_sites_skipper[['window_name','change_true']].merge(Flipper_results[['window_name','change_predict']], on='window_name',how='left'))

merged["change_predict"] = merged["change_predict"].fillna("no_change")

In [8]:
truth = merged["change_true"]
call = merged["change_predict"]

merged["result"] = np.select(
    [(truth == "no_change") & (call == "no_change"),(truth == "no_change") & (call != "no_change"),
     (truth != "no_change") & (call == "no_change"), (truth == call) & (truth != "no_change"),
     (truth != call) & (truth != "no_change") & (call != "no_change")],
    ["TN","FP","FN","TP","FP",], default="ERROR")

## Accuracy measures.

In [10]:
tp = sum(merged["result"] == "TP")
fp = sum(merged["result"] == "FP")
tn = sum(merged["result"] == "TN")
fn = sum(merged["result"] == "FN")
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

print(f"Precision (fraction of true positive captured): {precision:.3f} ({tp}/{(tp+fp)})")

Precision (fraction of true positive captured): 0.991 (1005/1014)


In [11]:
sensitivity = tp/(tp+fn)

print(f"sensitivity: {sensitivity:.3f} ({tp}/{(tp+fn)})")

sensitivity: 0.222 (1005/4521)


In [12]:
specificity = tn/(tn+fp)

print(f"specificity: {specificity:.3f} ({tn}/{(tn+fp)})")

specificity: 0.998 (3975/3984)


# Ad-hoc method validation

In [13]:
skipper_control = pd.read_csv(f"{sim_dir}/skipper/reproducible_enriched_windows/control.reproducible_enriched_windows.tsv.gz",sep = "\t")
skipper_treatment = pd.read_csv(f"{sim_dir}/skipper/reproducible_enriched_windows/treatment.reproducible_enriched_windows.tsv.gz",sep = "\t")

In [14]:
control_windows   = set(skipper_control['window_name'])
treatment_windows = set(skipper_treatment['window_name'])

# In control but not in treatment. 
only_in_control = skipper_control[
    ~skipper_control['window_name'].isin(treatment_windows)
]
only_in_control["change_predict"] = "decreased"
only_in_control = only_in_control[["window_name","change_predict"]]

# In treatment but not in control.
only_in_treatment = skipper_treatment[
    ~skipper_treatment['window_name'].isin(control_windows)
]
only_in_treatment["change_predict"] = "increased"
only_in_treatment = only_in_treatment[["window_name","change_predict"]]

# In both treatment and control.
in_both = pd.merge(
    skipper_control,
    skipper_treatment,
    on='window_name',
    suffixes=('_control', '_treatment')
)
in_both["change_predict"] = "no_change"
in_both = in_both[["window_name","change_predict"]]

ad_hoc_results = pd.concat([only_in_control, only_in_treatment, in_both], axis = 0, ignore_index = True)

In [15]:
# Merge truth and results.
merged = (true_sites_skipper[['window_name','change_true']].merge(ad_hoc_results[['window_name','change_predict']], on='window_name', how='left'))

In [16]:
truth = merged["change_true"]
call = merged["change_predict"]

merged["result"] = np.select(
    [(truth == "no_change") & (call == "no_change"),(truth == "no_change") & (call != "no_change"),
     (truth != "no_change") & (call == "no_change"), (truth == call) & (truth != "no_change"),
     (truth != call) & (truth != "no_change") & (call != "no_change")],
    ["TN","FP","FN","TP","FP",], default="ERROR"
)

## Accuracy measures.

In [17]:
tp = sum(merged["result"] == "TP")
fp = sum(merged["result"] == "FP")
tn = sum(merged["result"] == "TN")
fn = sum(merged["result"] == "FN")
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

print(f"Precision: {precision:.3f} ({tp}/{(tp+fp)})")

Precision: 0.558 (3237/5803)


In [18]:
sensitivity = tp/(tp+fn)

print(f"sensitivity: {sensitivity:.3f} ({tp}/{(tp+fn)})")

sensitivity: 0.726 (3237/4459)


In [19]:
specificity = tn/(tn+fp)

print(f"specificity: {specificity:.3f} ({tn}/{(tn+fp)})")

specificity: 0.366 (1480/4046)


# Differential skipper validation 

In [20]:
dif_skip_up = pd.read_csv(f"{sim_dir}/diff_skipper/reproducible_enriched_windows/ctr-trt.reproducible_enriched_windows.tsv",sep = "\t")
dif_skip_down = pd.read_csv(f"{sim_dir}/diff_skipper/reproducible_enriched_windows/trt-ctr.reproducible_enriched_windows.tsv",sep = "\t")
dif_skip_up["change_predict"] = "increased"
dif_skip_down["change_predict"] = "decreased"
dif_skip_results = pd.concat([dif_skip_up, dif_skip_down], axis = 0, ignore_index = True)

In [21]:
# Merge truth and results.
merged = (true_sites_skipper[['window_name','change_true']].merge(dif_skip_results[['window_name','change_predict']], on='window_name', how='left'))
merged['change_predict'] = merged['change_predict'].fillna('no_change')

In [22]:
truth = merged["change_true"]
call = merged["change_predict"]

merged["result"] = np.select(
    [(truth == "no_change") & (call == "no_change"),(truth == "no_change") & (call != "no_change"),
     (truth != "no_change") & (call == "no_change"), (truth == call) & (truth != "no_change"),
     (truth != call) & (truth != "no_change") & (call != "no_change")],
    ["TN","FP","FN","TP","FP",], default="ERROR"
)

## Accuracy measures.

In [23]:
tp = sum(merged["result"] == "TP")
fp = sum(merged["result"] == "FP")
tn = sum(merged["result"] == "TN")
fn = sum(merged["result"] == "FN")
precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

print(f"Precision: {precision:.3f} ({tp}/{tp+fp})")

Precision: 0.997 (775/777)


In [24]:
sensitivity = tp/(tp+fn)

print(f"sensitivity: {sensitivity:.3f} ({tp}/{(tp+fn)})")

sensitivity: 0.171 (775/4521)


In [25]:
specificity = tn/(tn+fp)

print(f"specificity: {specificity:.3f} ({tn}/{(tn+fp)})")

specificity: 0.999 (3982/3984)
